In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz

In [ ]:
ctd= pd.read_csv("./adr_ctd_disease_genes.csv")
# ctd.head()

,adr_id,adr_name,CTD_DiseaseID,GeneSymbol,GeneID,source
0,C0002170,Alopecia,MESH:D000505,ABCC2,1244,CTD_disease
1,C0002170,Alopecia,MESH:D000505,AR,367,CTD_disease
2,C0002170,Alopecia,MESH:D000505,BRD4,23476,CTD_disease
3,C0002170,Alopecia,MESH:D000505,CRH,1392,CTD_disease
4,C0002170,Alopecia,MESH:D000505,HR,55806,CTD_disease


In [7]:
sider= pd.read_csv("./sider_lincs_common_clean_FINAL.csv")
sider.head()

,drug_id,drug_cid,drug_name,sider_norm_name,pert_id,pert_iname,adr_id,adr_name
0,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0002170,Alopecia
1,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0232462,Decreased appetite
2,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009450,Infection
3,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009806,Constipation
4,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0011603,Dermatitis


In [16]:
hpo= pd.read_csv("./adr_hpo_gene_grounding_clean.csv")
hpo.head()

,adr_id,adr_name,HPO_ID,hpo_name,gene_symbol,ncbi_gene_id
0,C0002170,Alopecia,HP:0007534,Congenital posterior occipital alopecia,HRAS,3265
1,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT10,3858
2,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT1,3848
3,C0232462,Decreased appetite,HP:0004396,Poor appetite,IL18BP,10068
4,C0232462,Decreased appetite,HP:0004396,Poor appetite,TRANK1,9881


In [15]:
gene_lincs = pd.read_csv("gene_index_lincs.csv")
gene_lincs.head()

,gene_id,GeneSymbol
0,10007,GNPDA1
1,1001,CDH3
2,10013,HDAC6
3,10038,PARP2
4,10046,MAMLD1


In [31]:
# Normalize gene symbols
gene_lincs["GeneSymbol"] = gene_lincs["GeneSymbol"].str.upper().str.strip()
ctd["GeneSymbol"] = ctd["GeneSymbol"].str.upper().str.strip()
hpo["gene_symbol"] = hpo["gene_symbol"].str.upper().str.strip()

In [18]:
valid_genes = set(gene_lincs["GeneSymbol"])

len(valid_genes), \
len(set(ctd["GeneSymbol"]) & valid_genes), \
len(set(hpo["gene_symbol"]) & valid_genes)


(978, 445, 364)

In [25]:
gene_lincs = pd.read_csv("gene_index_lincs.csv")

gene_lincs["GeneSymbol"] = gene_lincs["GeneSymbol"].str.upper().str.strip()
gene_lincs = gene_lincs.reset_index(drop=True)
gene_lincs["gene_idx"] = gene_lincs.index

gene_lincs = gene_lincs[["gene_idx", "GeneSymbol"]]


In [26]:
adr_index = (
    sider[["adr_id", "adr_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

adr_index["adr_idx"] = adr_index.index

In [41]:
adr_index = adr_index[["adr_idx", "adr_id", "adr_name"]]

adr_index.to_csv("adr_index.csv", index=False)

In [27]:
ctd_m = (
    ctd
    .merge(adr_index[["adr_id", "adr_idx"]], on="adr_id", how="inner")
    .merge(gene_lincs, on="GeneSymbol", how="inner")
)

rows = ctd_m["adr_idx"].values
cols = ctd_m["gene_idx"].values
data = np.ones(len(ctd_m), dtype=np.int8)

Gadr_ctd = csr_matrix(
    (data, (rows, cols)),
    shape=(len(adr_index), len(gene_lincs))
)


In [28]:
hpo_m = (
    hpo
    .merge(adr_index[["adr_id", "adr_idx"]], on="adr_id", how="inner")
    .merge(
        gene_lincs,
        left_on="gene_symbol",
        right_on="GeneSymbol",
        how="inner"
    )
)

rows = hpo_m["adr_idx"].values
cols = hpo_m["gene_idx"].values
data = np.ones(len(hpo_m), dtype=np.int8)

Gadr_hpo = csr_matrix(
    (data, (rows, cols)),
    shape=(len(adr_index), len(gene_lincs))
)


In [29]:
Gadr = Gadr_ctd.maximum(Gadr_hpo)


In [30]:
assert Gadr.shape[0] == len(adr_index)
assert Gadr.shape[1] == len(gene_lincs)

print("Avg genes / ADR:", float(Gadr.sum(axis=1).mean()))
print("Max genes / ADR:", int(Gadr.sum(axis=1).max()))


Avg genes / ADR: 5.7103033615742005
Max genes / ADR: 319


In [32]:
n_adr, n_gene = Gadr.shape
nnz = Gadr.nnz

density = nnz / (n_adr * n_gene)

print("ADRs:", n_adr)
print("Genes:", n_gene)
print("Non-zeros:", nnz)
print("Density:", density)

ADRs: 3659
Genes: 978
Non-zeros: 20894
Density: 0.005838755993429653


In [33]:
genes_per_adr = np.array(Gadr.sum(axis=1)).flatten()

print("Mean genes / ADR:", genes_per_adr.mean())
print("Median genes / ADR:", np.median(genes_per_adr))
print("Max genes / ADR:", genes_per_adr.max())
print("Min genes / ADR:", genes_per_adr.min())

Mean genes / ADR: 5.7103033615742005
Median genes / ADR: 0.0
Max genes / ADR: 319
Min genes / ADR: 0


In [34]:
adrs_per_gene = np.array(Gadr.sum(axis=0)).flatten()

print("Mean ADRs / gene:", adrs_per_gene.mean())
print("Median ADRs / gene:", np.median(adrs_per_gene))
print("Max ADRs / gene:", adrs_per_gene.max())

Mean ADRs / gene: 21.3640081799591
Median ADRs / gene: 2.0
Max ADRs / gene: 348


In [35]:
print("CTD links:", Gadr_ctd.nnz)
print("HPO links:", Gadr_hpo.nnz)
print("Union links:", Gadr.nnz)

CTD links: 2546
HPO links: 18455
Union links: 20894


In [37]:
assert np.all(genes_per_adr >= 0)
assert np.all(adrs_per_gene >= 0)
assert Gadr.nnz > 0


In [38]:
save_npz("Gadr.npz", Gadr)

In [39]:
save_npz("Gadr_ctd.npz", Gadr_ctd)
save_npz("Gadr_hpo.npz", Gadr_hpo)

In [40]:
import json

summary = {
    "n_adr": int(Gadr.shape[0]),
    "n_gene": int(Gadr.shape[1]),
    "nnz": int(Gadr.nnz),
    "density": float(Gadr.nnz / (Gadr.shape[0] * Gadr.shape[1])),
    "mean_genes_per_adr": float(genes_per_adr.mean()),
    "median_genes_per_adr": float(np.median(genes_per_adr)),
    "max_genes_per_adr": int(genes_per_adr.max()),
}

with open("Gadr_summary.json", "w") as f:
    json.dump(summary, f, indent=2)